# Notebook 03 — Custom-50 Evaluation
**D7047E Advanced Deep Learning | Group 14**

Two-stage evaluation on the **custom-50** real-photo dataset:
1. **Binary**: best binary model (IAM test F1) → custom-50 (clean vs crossed-out)
2. **Multiclass**: best multiclass model (IAM test Macro-F1) → custom-50 (8 styles)

Generalisation gap = IAM test F1 − Custom-50 F1 for each task.

Run after `02_binary.ipynb` and `02_multiclass.ipynb` have completed.

WandB: `adl-crossouts-v3 / custom / {best_model_name}`

## 1. Configuration

In [26]:
import os
IN_COLAB = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_BACKEND_VERSION' in os.environ
DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')
if IN_COLAB and not DRIVE_MOUNTED:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_MOUNTED = os.path.exists('/content/drive/MyDrive')
    except Exception as e:
        print(f'Drive mount skipped ({e}). Checkpoints will be loaded from /content/adl_checkpoints.')
print(f'IN_COLAB={IN_COLAB}  DRIVE_MOUNTED={DRIVE_MOUNTED}')

IN_COLAB=False  DRIVE_MOUNTED=False


In [28]:
import sys, os
sys.path.insert(0, '..')
IN_COLAB       = 'COLAB_RELEASE_TAG' in os.environ or 'COLAB_BACKEND_VERSION' in os.environ
DRIVE_MOUNTED  = os.path.exists('/content/drive/MyDrive')
DATA_DIR       = '../dataset/iam_crossouts'
CUSTOM_DIR     = '../dataset/custom_50'
# CHECKPOINT_DIR = '../checkpoints'                              # local (original)
if DRIVE_MOUNTED:
    CHECKPOINT_DIR = '/content/drive/MyDrive/adl_checkpoints'   # Google Drive (persistent)
elif IN_COLAB:
    CHECKPOINT_DIR = '/content/adl_checkpoints'                  # Colab local (lost on restart)
else:
    CHECKPOINT_DIR = '../checkpoints'                            # local machine
IMG_SIZE       = 224
BATCH_SIZE     = 64
NUM_WORKERS    = 4        # Colab has 2 vCPUs; 4 is safe upper bound
WANDB_GROUP    = 'custom'
from common import CATEGORIES
NC = len(CATEGORIES)
print(f'Config loaded. NC={NC}  CHECKPOINT_DIR={CHECKPOINT_DIR}')

Config loaded. NC=8  CHECKPOINT_DIR=../checkpoints


## 2. Setup

In [6]:
!pip install -q gdown torch torchvision pillow matplotlib scikit-learn wandb


[notice] A new release of pip is available: 25.1.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import os
os.environ['WANDB_API_KEY'] =  "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"

In [30]:
import os, gc
import torch, torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score)
import wandb
from dotenv import load_dotenv

from common import get_transforms, WANDB_PROJECT, WANDB_GROUP_BINARY, WANDB_GROUP_MULTICLASS, CrossOutDataset, BinaryDataset, CATEGORIES, rebuild_model, rebuild_binary_model

load_dotenv()
wandb.login(key=os.environ.get('WANDB_API_KEY'))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/nagarajan.ganesan/.netrc


Device: cpu


## 3. Model Reconstruction Helpers
Used to rebuild model architectures before loading checkpoint weights.

In [32]:
from common import SimpleCNN, rebuild_model, rebuild_binary_model
print("Model helpers ready.")


Model helpers ready.


## 4a. Load All Multiclass Checkpoints & Pick Best

In [ ]:
_, val_t = get_transforms(IMG_SIZE)
ldr_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
              pin_memory=True, prefetch_factor=2 if NUM_WORKERS>0 else None)

# Load IAM test set for final comparison
test_ds     = CrossOutDataset(os.path.join(DATA_DIR,'test','images'), CATEGORIES, val_t)
test_loader = DataLoader(test_ds, shuffle=False, **ldr_kw)

# Scan checkpoints/ for multiclass .pth files
mc_files = [f for f in os.listdir(CHECKPOINT_DIR)
            if f.startswith('best_mc_') and f.endswith('.pth')]
print(f'Found {len(mc_files)} multiclass checkpoints: {mc_files}')

models_table = {}
for fname in sorted(mc_files):
    path = os.path.join(CHECKPOINT_DIR, fname)
    ckpt = torch.load(path, map_location=device)
    model_name = ckpt['model_name']
    model = rebuild_model(model_name, NC)
    model.load_state_dict(ckpt['model_state_dict'])
    model = model.to(device)
    model.eval()

    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in test_loader:
            preds.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
            labels.extend(lbs.tolist())

    f1 = f1_score(labels, preds, average='macro', zero_division=0)
    models_table[model_name] = {'f1': f1, 'preds': preds, 'labels': labels}
    print(f'  {model_name:<18} Macro-F1={f1:.4f} (checkpoint: {fname})')
    del model; torch.cuda.empty_cache(); gc.collect()

best_name = max(models_table, key=lambda k: models_table[k]['f1'])
print(f'\nBest multiclass model: {best_name} (Macro-F1={models_table[best_name]["f1"]:.4f})')

Found 1 multiclass checkpoints: ['best_mc_SimpleCNN.pth']


/var/folders/34/gp_zqq8s68l08cbyqd89g9x80000gn/T/ipykernel_77669/3348387261.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(path, map_location=device)

## 4b. Load All Binary Checkpoints & Pick Best

In [ ]:
# bin_test_ds     = BinaryDataset(os.path.join(DATA_DIR,'test','images'), val_t)
# bin_test_loader = DataLoader(bin_test_ds, shuffle=False, **ldr_kw)

# bin_files = [f for f in os.listdir(CHECKPOINT_DIR)
#              if f.startswith('best_binary_') and f.endswith('.pth')]
# print(f'Found {len(bin_files)} binary checkpoints: {bin_files}')

# bin_table = {}
# for fname in sorted(bin_files):
#     path = os.path.join(CHECKPOINT_DIR, fname)
#     ckpt = torch.load(path, map_location=device)
#     model_name = ckpt['model_name']
#     model = rebuild_binary_model(model_name)
#     model.load_state_dict(ckpt['model_state_dict'])
#     model = model.to(device)
#     model.eval()

#     preds, labels, probs = [], [], []
#     with torch.no_grad():
#         for imgs, lbs in bin_test_loader:
#             out = model(imgs.to(device))
#             probs.extend(torch.softmax(out,dim=1)[:,1].cpu().tolist())
#             preds.extend(out.argmax(1).cpu().tolist())
#             labels.extend(lbs.tolist())

#     f1 = f1_score(labels, preds, zero_division=0)
#     bin_table[model_name] = {'f1': f1, 'preds': preds, 'labels': labels, 'probs': probs}
#     print(f'  {model_name:<18} F1={f1:.4f} (checkpoint: {fname})')
#     del model; torch.cuda.empty_cache(); gc.collect()

# best_bin_name = max(bin_table, key=lambda k: bin_table[k]['f1'])
# print(f'\nBest binary model: {best_bin_name} (F1={bin_table[best_bin_name]["f1"]:.4f})')

## 5. Custom-50 Evaluation

In [ ]:
custom_ds     = CrossOutDataset(CUSTOM_DIR, CATEGORIES, val_t)
print("Crossout")
custom_loader = DataLoader(custom_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
print("
print(f'Custom-50 multiclass: {len(custom_ds)} images\n')

mc_c50_results = {}
for model_name in sorted(models_table.keys()):
    safe = model_name.replace('/','_').replace('-','_')
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'best_mc_{safe}.pth')
    model = rebuild_model(model_name, NC)
    model.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state_dict'])
    model = model.to(device)
    model.eval()
    
    preds, labels = [], []
    with torch.no_grad():
        for imgs, lbs in custom_loader:
            preds.extend(model(imgs.to(device)).argmax(1).cpu().tolist())
            labels.extend(lbs.tolist())

    c50_f1  = f1_score(labels, preds, average='macro', zero_division=0)
    c50_acc = accuracy_score(labels, preds)
    mc_c50_results[model_name] = {'f1': c50_f1, 'acc': c50_acc, 'preds': preds, 'labels': labels}
    print(f'  {model_name:<18} Acc={c50_acc:.4f}  Macro-F1={c50_f1:.4f}')
    del model; torch.cuda.empty_cache(); gc.collect()

best_name = max(mc_c50_results, key=lambda k: mc_c50_results[k]['f1'])
print(f'\nBest on Custom-50 (multiclass): {best_name}')
print()
print(classification_report(mc_c50_results[best_name]['labels'],
                             mc_c50_results[best_name]['preds'],
                             target_names=CATEGORIES, zero_division=0))

## 5b. Binary Custom-50 Evaluation

In [ ]:
# bin_custom_ds     = BinaryDataset(CUSTOM_DIR, val_t)
# bin_custom_loader = DataLoader(bin_custom_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
# print(f'Custom-50 binary: {len(bin_custom_ds)} images\n')

# bin_c50_results = {}
# for model_name in sorted(bin_table.keys()):
#     safe = model_name.replace('/','_').replace('-','_')
#     ckpt_path = os.path.join(CHECKPOINT_DIR, f'best_binary_{safe}.pth')
#     model = rebuild_binary_model(model_name)
#     model.load_state_dict(torch.load(ckpt_path, map_location=device)['model_state_dict'])
#     model = model.to(device)
#     model.eval()

#     preds, labels, probs = [], [], []
#     with torch.no_grad():
#         for imgs, lbs in bin_custom_loader:
#             out = model(imgs.to(device))
#             probs.extend(torch.softmax(out,dim=1)[:,1].cpu().tolist())
#             preds.extend(out.argmax(1).cpu().tolist())
#             labels.extend(lbs.tolist())

#     c50_f1  = f1_score(labels, preds, zero_division=0)
#     c50_acc = accuracy_score(labels, preds)
#     bin_c50_results[model_name] = {'f1': c50_f1, 'acc': c50_acc, 'preds': preds, 'labels': labels, 'probs': probs}
#     print(f'  {model_name:<18} Acc={c50_acc:.4f}  F1={c50_f1:.4f}')
#     del model; torch.cuda.empty_cache(); gc.collect()

# best_bin_name = max(bin_c50_results, key=lambda k: bin_c50_results[k]['f1'])
# print(f'\nBest on Custom-50 (binary): {best_bin_name}')
# print()
# print(classification_report(bin_c50_results[best_bin_name]['labels'],
#                              bin_c50_results[best_bin_name]['preds'],
#                              target_names=['CLEAN','CROSSED'], zero_division=0))

## 7. Comparison Plots

In [24]:
# print('=== Generalisation Gap — Binary (IAM Test F1 vs Custom-50 F1) ===')
# print(f'{"Model":<18} {"IAM Test F1":>12} {"Custom-50 F1":>13} {"Gap":>8}')
# print('-' * 55)
# for name in sorted(bin_table.keys()):
#     iam = bin_table[name]['f1']
#     c50 = bin_c50_results[name]['f1']
#     marker = ' <-- best' if name == best_bin_name else ''
#     print(f'{name:<18} {iam:>12.4f} {c50:>13.4f} {iam-c50:>+8.4f}{marker}')

# print()
print('=== Generalisation Gap — Multiclass (IAM Test Macro-F1 vs Custom-50 Macro-F1) ===')
print(f'{"Model":<18} {"IAM Test F1":>12} {"Custom-50 F1":>13} {"Gap":>8}')
print('-' * 55)
for name in sorted(models_table.keys()):
    iam = models_table[name]['f1']
    c50 = mc_c50_results[name]['f1']
    marker = ' <-- best' if name == best_name else ''
    print(f'{name:<18} {iam:>12.4f} {c50:>13.4f} {iam-c50:>+8.4f}{marker}')

def _cm_figure(labels, preds, class_names, title):
    present = sorted(set(labels))
    names   = [class_names[i] for i in present]
    cm = confusion_matrix(labels, preds, labels=present)
    fig, ax = plt.subplots(figsize=(max(5, len(names)*1.2), max(4, len(names))))
    ConfusionMatrixDisplay(cm, display_labels=names).plot(
        ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues')
    ax.set_title(title)
    plt.tight_layout()
    return fig

def _per_class_metrics(labels, preds, class_names):
    metrics = {}
    present = sorted(set(labels))
    for i in present:
        cat = class_names[i]
        lbl_i = [1 if l == i else 0 for l in labels]
        prd_i = [1 if p == i else 0 for p in preds]
        metrics[f'{cat}_precision'] = precision_score(lbl_i, prd_i, zero_division=0)
        metrics[f'{cat}_recall']    = recall_score(lbl_i, prd_i, zero_division=0)
        metrics[f'{cat}_f1']        = f1_score(lbl_i, prd_i, zero_division=0)
    return metrics

# WandB — log all binary models
for name in bin_table:
    run = wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, name=f'binary_{name}',
                     config=dict(model=name, task='custom50_binary'), reinit=True)
    r = bin_c50_results[name]
    wandb.log({
        'iam_test_f1':        bin_table[name]['f1'],
        'custom50_acc':       r['acc'],
        'custom50_f1':        r['f1'],
        'custom50_precision': precision_score(r['labels'], r['preds'], zero_division=0),
        'custom50_recall':    recall_score(r['labels'], r['preds'], zero_division=0),
        'generalisation_gap': bin_table[name]['f1'] - r['f1'],
        'confusion_matrix':   wandb.Image(_cm_figure(
            r['labels'], r['preds'], ['CLEAN', 'CROSSED'],
            f'Custom-50 Binary CM — {name}')),
    })
    plt.close('all')
    run.finish()

# WandB — log all multiclass models
for name in models_table:
    run = wandb.init(project=WANDB_PROJECT, group=WANDB_GROUP, name=f'multiclass_{name}',
                     config=dict(model=name, task='custom50_multiclass'), reinit=True)
    r = mc_c50_results[name]
    log_dict = {
        'iam_test_f1':        models_table[name]['f1'],
        'custom50_acc':       r['acc'],
        'custom50_macro_f1':  r['f1'],
        'custom50_precision': precision_score(r['labels'], r['preds'], average='macro', zero_division=0),
        'custom50_recall':    recall_score(r['labels'], r['preds'], average='macro', zero_division=0),
        'generalisation_gap': models_table[name]['f1'] - r['f1'],
        'confusion_matrix':   wandb.Image(_cm_figure(
            r['labels'], r['preds'], CATEGORIES,
            f'Custom-50 Multiclass CM — {name}')),
    }
    log_dict.update(_per_class_metrics(r['labels'], r['preds'], CATEGORIES))
    wandb.log(log_dict)
    plt.close('all')
    run.finish()

print('All results logged to WandB.')

=== Generalisation Gap — Multiclass (IAM Test Macro-F1 vs Custom-50 Macro-F1) ===
Model               IAM Test F1  Custom-50 F1      Gap
-------------------------------------------------------


NameError: name 'bin_table' is not defined

## 9. Sample Predictions Grid — Best Multiclass Model (Custom-50)

In [ ]:
model_names = sorted(models_table.keys())
bin_names   = sorted(bin_table.keys())
x = np.arange(len(model_names))
w = 0.35

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Multiclass ---
mc_iam = [models_table[n]['f1']    for n in model_names]
mc_c50 = [mc_c50_results[n]['f1'] for n in model_names]
bars1 = axes[0].bar(x - w/2, mc_iam, w, label='IAM Test',   color='#5c85d6')
bars2 = axes[0].bar(x + w/2, mc_c50, w, label='Custom-50',  color='#e07b39')
axes[0].set_xticks(x); axes[0].set_xticklabels(model_names, rotation=15, ha='right')
axes[0].set_ylim(0, 1); axes[0].set_ylabel('Macro-F1')
axes[0].set_title('Multiclass — IAM Test vs Custom-50')
axes[0].legend()
for bar in bars1: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                                f'{bar.get_height():.2f}', ha='center', fontsize=8)
for bar in bars2: axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                                f'{bar.get_height():.2f}', ha='center', fontsize=8)

# --- Binary ---
bx = np.arange(len(bin_names))
bin_iam = [bin_table[n]['f1']    for n in bin_names]
bin_c50 = [bin_c50_results[n]['f1'] for n in bin_names]
bars3 = axes[1].bar(bx - w/2, bin_iam, w, label='IAM Test',  color='#5c85d6')
bars4 = axes[1].bar(bx + w/2, bin_c50, w, label='Custom-50', color='#e07b39')
axes[1].set_xticks(bx); axes[1].set_xticklabels(bin_names, rotation=15, ha='right')
axes[1].set_ylim(0, 1); axes[1].set_ylabel('F1')
axes[1].set_title('Binary — IAM Test vs Custom-50')
axes[1].legend()
for bar in bars3: axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                                f'{bar.get_height():.2f}', ha='center', fontsize=8)
for bar in bars4: axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                                f'{bar.get_height():.2f}', ha='center', fontsize=8)

plt.suptitle('Generalisation: IAM Test vs Custom-50 (all models)', fontsize=12)
plt.tight_layout(); plt.savefig('custom50_comparison.png', dpi=150); plt.show()
print('Saved: custom50_comparison.png')

## 8. Confusion Matrix — Best Multiclass Model (Custom-50)

## 8b. Confusion Matrix — Best Binary Model (Custom-50)

In [ ]:
cm_bin = confusion_matrix(bin_c50_results[best_bin_name]['labels'],
                          bin_c50_results[best_bin_name]['preds'])
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm_bin, display_labels=['CLEAN', 'CROSSED']).plot(
    ax=ax, colorbar=False, cmap='Blues')
plt.title(f'Custom-50 Binary Confusion Matrix — {best_bin_name}')
plt.tight_layout(); plt.savefig('custom50_binary_cm.png', dpi=150); plt.show()
print('Saved: custom50_binary_cm.png')

In [ ]:
best_mc_labels = mc_c50_results[best_name]['labels']
best_mc_preds  = mc_c50_results[best_name]['preds']
present        = sorted(set(best_mc_labels))
present_names  = [CATEGORIES[i] for i in present]
cm = confusion_matrix(best_mc_labels, best_mc_preds, labels=present)
fig, ax = plt.subplots(figsize=(9, 7))
ConfusionMatrixDisplay(cm, display_labels=present_names).plot(
    ax=ax, xticks_rotation=45, colorbar=False, cmap='Blues')
plt.title(f'Custom-50 Confusion Matrix — {best_name}')
plt.tight_layout(); plt.savefig('custom50_cm.png', dpi=150); plt.show()
print('Saved: custom50_cm.png')

## 8c. Confusion Matrices — All Multiclass Models (Custom-50)

In [ ]:
n_models = len(mc_c50_results)
cols = min(n_models, 3)
rows = (n_models + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(cols * 6, rows * 5))
axes = axes.flatten() if n_models > 1 else [axes]

for i, model_name in enumerate(sorted(mc_c50_results.keys())):
    res    = mc_c50_results[model_name]
    present       = sorted(set(res['labels']))
    present_names = [CATEGORIES[j] for j in present]
    cm = confusion_matrix(res['labels'], res['preds'], labels=present)
    f1 = f1_score(res['labels'], res['preds'], average='macro', zero_division=0)
    ConfusionMatrixDisplay(cm, display_labels=present_names).plot(
        ax=axes[i], xticks_rotation=45, colorbar=False, cmap='Blues')
    marker = ' ★' if model_name == best_name else ''
    axes[i].set_title(f'{model_name}{marker}\nMacro-F1={f1:.3f}', fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Custom-50 Confusion Matrices — All Multiclass Models', fontsize=12)
plt.tight_layout()
plt.savefig('custom50_all_mc_cm.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: custom50_all_mc_cm.png')

## 8. Sample Predictions Grid

In [ ]:
IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])
N_SAMPLES = 3

# Reload best multiclass model for inference
best_ckpt_path = os.path.join(CHECKPOINT_DIR,
    f'best_mc_{best_name.replace("/","_").replace("-","_")}.pth')
best_model = rebuild_model(best_name, NC)
best_model.load_state_dict(torch.load(best_ckpt_path, map_location=device)['model_state_dict'])
best_model = best_model.to(device)
best_model.eval()

from PIL import Image
fig, axes = plt.subplots(len(CATEGORIES), N_SAMPLES, figsize=(N_SAMPLES*3, len(CATEGORIES)*2))

for row, cat in enumerate(CATEGORIES):
    folder = os.path.join(CUSTOM_DIR, cat)
    if not os.path.exists(folder):
        for col in range(N_SAMPLES): axes[row][col].axis('off')
        continue
    files = sorted([f for f in os.listdir(folder) if f.endswith('.png')])[:N_SAMPLES]
    for col, fname in enumerate(files):
        raw    = Image.open(os.path.join(folder, fname)).convert('RGB')
        tensor = val_t(raw).unsqueeze(0).to(device)
        with torch.no_grad():
            probs    = torch.softmax(best_model(tensor), dim=1)[0]
            pred_idx = probs.argmax().item()
        pred_cat = CATEGORIES[pred_idx]; conf = probs[pred_idx].item()
        disp = (val_t(raw) * IMAGENET_STD[:,None,None] + IMAGENET_MEAN[:,None,None]).clamp(0,1)
        axes[row][col].imshow(disp.permute(1,2,0).numpy()); axes[row][col].axis('off')
        color = 'green' if pred_cat == cat else 'red'
        axes[row][col].set_title(f'{pred_cat}\n{conf:.0%}', fontsize=7, color=color)
    for col in range(len(files), N_SAMPLES):
        axes[row][col].axis('off')
    axes[row][0].set_ylabel(f'GT: {cat}', fontsize=8, rotation=0, labelpad=65, va='center')

fig.suptitle(f'Custom-50 predictions — {best_name} (green=correct, red=wrong)', fontsize=11)
plt.tight_layout(); plt.savefig('custom50_predictions.png', dpi=150); plt.show()
print('Saved: custom50_predictions.png')
del best_model; torch.cuda.empty_cache(); gc.collect()

In [ ]:

# ── Misclassified examples ───────────────────────────────────────────────────
# Shows ONLY the images the model got wrong, with true label and predicted label.

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406])
IMAGENET_STD  = torch.tensor([0.229, 0.224, 0.225])

# Reload best multiclass model
best_ckpt_path = os.path.join(CHECKPOINT_DIR,
    f'best_mc_{best_name.replace("/","_").replace("-","_")}.pth')
best_model = rebuild_model(best_name, NC)
best_model.load_state_dict(torch.load(best_ckpt_path, map_location=device)['model_state_dict'])
best_model = best_model.to(device).eval()

# Collect all (image_path, true_class, pred_class, confidence)
wrong = []
for cat_idx, cat in enumerate(CATEGORIES):
    folder = os.path.join(CUSTOM_DIR, cat)
    if not os.path.exists(folder):
        continue
    for fname in sorted(os.listdir(folder)):
        if not fname.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue
        img_path = os.path.join(folder, fname)
        raw    = Image.open(img_path).convert('RGB')
        tensor = val_t(raw).unsqueeze(0).to(device)
        with torch.no_grad():
            probs    = torch.softmax(best_model(tensor), dim=1)[0]
            pred_idx = probs.argmax().item()
        if pred_idx != cat_idx:
            wrong.append((img_path, cat, CATEGORIES[pred_idx], probs[pred_idx].item(), raw))

print(f'{best_name} — {len(wrong)} misclassified out of {len(custom_ds)} Custom-50 images\n')
for img_path, true_cls, pred_cls, conf, _ in wrong:
    print(f'  True: {true_cls:<14}  Pred: {pred_cls:<14}  Conf: {conf:.2f}  '
          f'File: {os.path.basename(img_path)}')

# Plot grid
if wrong:
    cols = min(5, len(wrong))
    rows = (len(wrong) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.8, rows * 3.4))
    axes = np.array(axes).flatten()

    for ax, (img_path, true_cls, pred_cls, conf, raw) in zip(axes, wrong):
        disp = (val_t(raw) * IMAGENET_STD[:,None,None] + IMAGENET_MEAN[:,None,None]).clamp(0,1)
        ax.imshow(disp.permute(1,2,0).numpy())
        ax.set_title(f'True:  {true_cls}\nPred:  {pred_cls}\nConf: {conf:.0%}',
                     fontsize=7, color='red')
        ax.axis('off')
        for spine in ax.spines.values():
            spine.set_edgecolor('red'); spine.set_linewidth(2)

    for ax in axes[len(wrong):]:
        ax.set_visible(False)

    plt.suptitle(f'Misclassified Examples — {best_name} on Custom-50 ({len(wrong)} errors)',
                 fontsize=10, fontweight='bold')
    plt.tight_layout()
    plt.savefig('custom50_misclassified.png', dpi=150)
    plt.show()
    print('Saved: custom50_misclassified.png')

del best_model; torch.cuda.empty_cache(); gc.collect()
